# NLP Pipeline — Postgraduate AI Deliverables
### Image OCR → Text Cleaning → Representation → NER → Key Phrases → Section Categorisation
---

## 0. Setup & Imports

In [ ]:
# ── Image / OCR ───────────────────────────────────────────────────────────
import cv2
import pytesseract

# ── Core NLP ──────────────────────────────────────────────────────────────
import spacy
from spacy import displacy
from IPython.display import display, HTML

# ── Text representation ───────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
import numpy as np

# ── Visualisation ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── NLTK (stemming / lemmatisation comparison) ────────────────────────────
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)

# Load spaCy model (same one used in class)
nlp = spacy.load('en_core_web_sm')

print('All libraries loaded successfully.')

---
## 1. Image Input — OCR with OpenCV & Tesseract
Reads a book/document image and extracts the raw text string, then shows
basic token counts — mirroring your class code exactly.

In [ ]:
# ── Load image with OpenCV ────────────────────────────────────────────────
# Change 'MyBook.jpg' to your own image file name.
# The file must be in the same folder as this notebook.
IMAGE_FILE = 'MyBook.jpg'

img = cv2.imread(IMAGE_FILE)

if img is None:
    raise FileNotFoundError(
        f"Could not load '{IMAGE_FILE}'. "
        "Make sure the image is in the same directory as this notebook."
    )

print('Image loaded successfully.')
print(f'Shape (height, width, channels): {img.shape}')

# ── Display the image inline ──────────────────────────────────────────────
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # OpenCV loads BGR; matplotlib needs RGB
plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)
plt.axis('off')
plt.title(f'Input image: {IMAGE_FILE}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── OCR with Tesseract (your class config) ────────────────────────────────
# --oem 1  = LSTM neural network engine (Tesseract 5)
# --psm 6  = assume a uniform block of text
custom_config = r'--oem 1 --psm 6'

# Extract raw text from image
ocr_text = pytesseract.image_to_string(img, config=custom_config)

print('=== RAW OCR OUTPUT ===')
print(ocr_text)

In [ ]:
# ── Pass OCR text through spaCy — same pattern as your class code ─────────
doc = nlp(ocr_text)

# Remove stop words and punctuation (your class code)
filtered_tokens = [token.text for token in doc if not token.is_stop and not token.is_punct]
clean_text = " ".join(filtered_tokens)

print('=== CLEANED TEXT (stopwords + punctuation removed) ===')
print(clean_text)

In [ ]:
# ── Token counts — your class code ───────────────────────────────────────
all_tokens      = [token.text for token in doc if not token.is_punct and not token.is_space]
filtered_tokens = [token.text for token in doc if not token.is_stop
                   and not token.is_punct and not token.is_space]

total_count    = len(all_tokens)
filtered_count = len(filtered_tokens)
stopword_count = total_count - filtered_count

print('Total tokens (with stopwords):', total_count)
print('Tokens (without stopwords):   ', filtered_count)
print('Stopwords removed:            ', stopword_count)

# ── Bar chart visualisation ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(
    ['All tokens', 'After stopword\nremoval', 'Stopwords\nremoved'],
    [total_count, filtered_count, stopword_count],
    color=['steelblue', 'mediumseagreen', 'coral'],
    edgecolor='white', width=0.5
)
ax.bar_label(bars, padding=4, fontsize=11, fontweight='bold')
ax.set_title(f'Token counts — {IMAGE_FILE}', fontsize=12)
ax.set_ylabel('Count')
ax.set_ylim(0, max(total_count, 1) * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feed OCR text into the corpus used by the rest of the pipeline ────────
#
# The OCR output is treated as a 'body' document and prepended to the
# hand-crafted corpus below.  All later sections (TF-IDF, NER, etc.)
# will include your scanned image text automatically.
#
# If ocr_text is empty (e.g. image not loaded), the corpus falls back
# to the built-in examples only.

ocr_document = (ocr_text.strip(), 'body')   # treat OCR output as a body section
print('OCR document ready — will be merged into corpus in Section 2.')

---
## 2. Sample Corpus
The OCR text from your image is automatically prepended as Doc 0.
The remaining documents are hand-crafted examples covering all three section types.

In [ ]:
# ── Corpus: list of (text, section_type) tuples ───────────────────────────
# Section types: 'title', 'caption', 'body'

documents_base = [
    ("Advances in Deep Learning for Natural Language Processing",
     "title"),

    ("""Natural language processing (NLP) has seen remarkable advances in recent years,
     driven primarily by transformer-based architectures such as BERT and GPT.
     Researchers at Google, OpenAI and universities across Europe have published
     hundreds of papers demonstrating state-of-the-art performance on tasks
     including sentiment analysis, named entity recognition, and machine translation.""",
     "body"),

    ("Figure 1: Accuracy of BERT vs baseline models on the SQuAD 2.0 benchmark.",
     "caption"),

    ("Applications of Machine Learning in Healthcare",
     "title"),

    ("""Machine learning models are increasingly being deployed in clinical settings.
     Hospitals in the United States and the United Kingdom have adopted AI tools
     for early detection of diseases such as cancer and diabetes.
     Companies like IBM, Microsoft and DeepMind have invested heavily
     in healthcare AI research throughout 2023 and 2024.""",
     "body"),

    ("Table 2: Comparison of diagnostic accuracy across five machine learning models.",
     "caption"),

    ("Climate Change and Renewable Energy Policy in Europe",
     "title"),

    ("""The European Union announced an ambitious €500 billion green energy programme
     in Brussels last March. Germany, France and Ireland pledged significant reductions
     in carbon emissions. The International Energy Agency reported that solar and wind
     capacity increased by 40% during this period.""",
     "body"),

    ("Figure 3: Renewable energy capacity growth across EU member states, 2020–2024.",
     "caption"),
]

# ── Prepend OCR document (from Section 1) if it has content ──────────────
if ocr_document[0]:                          # only add if OCR actually returned text
    documents = [ocr_document] + documents_base
else:
    documents = documents_base
    print('Note: OCR text was empty — using built-in corpus only.')

texts  = [d[0] for d in documents]
labels = [d[1] for d in documents]

print(f'{len(documents)} documents loaded (including OCR image if available).')
for i, (t, l) in enumerate(documents):
    preview = t[:70].replace('\n', ' ') + ('...' if len(t) > 70 else '')
    print(f'  [{i}] [{l.upper():7}] {preview}')

---
## 3. Text-Cleaning Pipeline
### 3a. Tokenisation (spaCy — same approach as class code)

In [ ]:
# ── Tokenise the first body paragraph ────────────────────────────────────
# Uses the first 'body' doc — this will be the OCR text if it loaded,
# otherwise the first built-in body paragraph.
body_indices = [i for i, l in enumerate(labels) if l == 'body']
sample_text  = texts[body_indices[0]]
doc = nlp(sample_text)

print('=== RAW TOKENS ===')
tokens = [token.text for token in doc]
print(' | '.join(tokens))

print(f'\nTotal tokens : {len(tokens)}')

### 3b. Stopword Removal

In [ ]:
# ── Remove stopwords, punctuation and whitespace ──────────────────────────
def remove_stopwords(doc):
    """Return tokens that are not stopwords, punctuation, or whitespace."""
    return [
        token.text.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
    ]

filtered = remove_stopwords(doc)

print('=== AFTER STOPWORD REMOVAL ===')
print(' | '.join(filtered))
print(f'\nOriginal token count : {len(tokens)}')
print(f'After filtering      : {len(filtered)}')
print(f'Tokens removed       : {len(tokens) - len(filtered)}')

### 3c. Stemming vs Lemmatisation — side-by-side comparison

In [ ]:
# ── Stemming (NLTK Porter Stemmer) ───────────────────────────────────────
stemmer   = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

# ── Lemmatisation (spaCy) — uses POS context for better accuracy ──────────
def lemmatise_tokens(doc_obj):
    return [
        token.lemma_.lower()
        for token in doc_obj
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
    ]

stemmed    = stem_tokens(filtered)
lemmatised = lemmatise_tokens(doc)

# ── Print comparison table ────────────────────────────────────────────────
print(f'{"Original":<22} {"Stemmed":<22} {"Lemmatised (spaCy)":<22}')
print('-' * 66)
for orig, stem, lemma in zip(filtered, stemmed, lemmatised):
    print(f'{orig:<22} {stem:<22} {lemma:<22}')

### 3d. Full Cleaning Pipeline — applied to all documents

In [ ]:
def clean_text(text):
    """
    Full pipeline:
      1. Tokenise with spaCy
      2. Lowercase
      3. Remove stopwords, punctuation, whitespace
      4. Lemmatise
      5. Keep only alphabetic tokens (removes stray numbers/symbols)
    Returns a cleaned string (space-joined lemmas).
    """
    doc = nlp(text)
    cleaned = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and token.is_alpha
    ]
    return ' '.join(cleaned)

cleaned_texts = [clean_text(t) for t in texts]

print('=== CLEANED TEXTS ===')
for i, (raw, cleaned, label) in enumerate(zip(texts, cleaned_texts, labels)):
    print(f'\n[Doc {i}] [{label.upper()}]')
    print(f'  Raw     : {raw[:90]}...' if len(raw) > 90 else f'  Raw     : {raw}')
    print(f'  Cleaned : {cleaned}')

---
## 4. Text Representation
### 4a. Bag of Words (CountVectorizer)

In [ ]:
# ── Bag of Words ─────────────────────────────────────────────────────────
bow_vectorizer = CountVectorizer(max_features=20)
bow_matrix     = bow_vectorizer.fit_transform(cleaned_texts)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out(),
    index=[f'Doc{i} [{labels[i]}]' for i in range(len(texts))]
)

print('=== BAG OF WORDS MATRIX (top 20 terms) ===')
display(bow_df)

# ── Visualise top-20 term frequencies (across all docs) ──────────────────
term_freq = bow_df.sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(term_freq.index, term_freq.values, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_title('Bag of Words — Top 20 Term Frequencies (all documents)', fontsize=13)
ax.set_xlabel('Term')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4b. TF-IDF Representation

In [ ]:
# ── TF-IDF ────────────────────────────────────────────────────────────────
tfidf_vectorizer = TfidfVectorizer(max_features=20)
tfidf_matrix     = tfidf_vectorizer.fit_transform(cleaned_texts)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f'Doc{i} [{labels[i]}]' for i in range(len(texts))]
)

print('=== TF-IDF MATRIX (top 20 terms) ===')
display(tfidf_df)

# ── Heatmap of TF-IDF scores ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(tfidf_df.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(tfidf_df.columns)))
ax.set_xticklabels(tfidf_df.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(tfidf_df.index)))
ax.set_yticklabels(tfidf_df.index, fontsize=9)
ax.set_title('TF-IDF Heatmap — Term Importance per Document', fontsize=13)
plt.colorbar(im, ax=ax, label='TF-IDF score')
plt.tight_layout()
plt.show()

---
## 5. Named Entity Recognition (NER)
### 5a. Entity extraction using spaCy (same API as class code)

In [ ]:
# ── Reuse the show_entity_info function from class code ───────────────────
def show_entity_info(doc_object):
    """Print entity text, label and description — from class code."""
    if doc_object.ents:
        header = f'{"Entity":<30} {"Label":<20} {"Description"}'
        print(header)
        print('-' * 70)
        for entity in doc_object.ents:
            print(f'{entity.text:<30} {entity.label_:<20} {spacy.explain(entity.label_)}')
    else:
        print('No entities found in text.')

# ── Run NER over every document ───────────────────────────────────────────
all_entities = []   # collect for visualisation later

for i, (text, label) in enumerate(zip(texts, labels)):
    doc_obj = nlp(text)
    if doc_obj.ents:
        print(f'\n=== Doc {i} [{label.upper()}] ===')
        show_entity_info(doc_obj)
        for ent in doc_obj.ents:
            all_entities.append({'doc': i, 'label': label, 'entity': ent.text, 'type': ent.label_})

entities_df = pd.DataFrame(all_entities)
print(f'\nTotal entities found across corpus: {len(entities_df)}')

### 5b. NER visualisation with displaCy (same approach as class code)

In [ ]:
# ── Visualise NER for the body paragraphs using displaCy ─────────────────
body_texts  = [t for t, l in zip(texts, labels) if l == 'body']

for body in body_texts:
    doc_obj = nlp(body)
    html = displacy.render(doc_obj, style='ent', jupyter=False)
    display(HTML(html))
    print()

### 5c. Entity frequency chart

In [ ]:
# ── Entity type distribution ──────────────────────────────────────────────
type_counts = entities_df['type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: entity type bar chart
axes[0].barh(type_counts.index, type_counts.values, color='coral', edgecolor='white')
axes[0].set_title('Entity Type Frequency (all docs)', fontsize=12)
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# Right: top individual entities
top_ents = entities_df['entity'].value_counts().head(15)
axes[1].barh(top_ents.index, top_ents.values, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Top 15 Named Entities', fontsize=12)
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 6. Key Phrase Extraction
### 6a. Noun chunks — same API as class code

In [ ]:
# ── Extract noun chunks (key phrases) from each body document ─────────────
# Uses the same noun_chunk API demonstrated in class

def extract_key_phrases(text):
    """Return noun chunks as key phrases with their dependency info."""
    doc_obj = nlp(text)
    phrases = []
    for chunk in doc_obj.noun_chunks:
        phrases.append({
            'phrase'      : chunk.text,
            'root_text'   : chunk.root.text,
            'root_dep'    : spacy.explain(chunk.root.dep_),
            'root_head'   : chunk.root.head.text
        })
    return phrases

# ── Print in the same tabular style as class code ─────────────────────────
for i, (text, label) in enumerate(zip(texts, labels)):
    phrases = extract_key_phrases(text)
    if phrases:
        print(f'\n=== Doc {i} [{label.upper()}] Key Phrases ===')
        print(f'{"Phrase":<35} {"Root":<18} {"Dependency":<28} {"Head"}')
        print('-' * 90)
        for p in phrases:
            print(f"{p['phrase']:<35} {p['root_text']:<18} {str(p['root_dep']):<28} {p['root_head']}")

### 6b. TF-IDF top key phrases per document

In [ ]:
# ── Top 5 TF-IDF key terms per document ──────────────────────────────────
tfidf_full   = TfidfVectorizer(max_features=100)
tfidf_full_m = tfidf_full.fit_transform(cleaned_texts)
feature_names = tfidf_full.get_feature_names_out()

print('=== TOP 5 KEY TERMS PER DOCUMENT (TF-IDF) ===')
for i, row in enumerate(tfidf_full_m.toarray()):
    top_idx   = row.argsort()[::-1][:5]
    top_terms = [(feature_names[j], round(row[j], 3)) for j in top_idx if row[j] > 0]
    if top_terms:
        print(f'\nDoc {i} [{labels[i].upper()}]:')
        for term, score in top_terms:
            print(f'  {term:<25} score: {score}')

### 6c. Key phrase frequency chart

In [ ]:
# ── Visualise key phrase importance with a simple frequency bar chart ─────
# (wordcloud library may not be installed in all environments;
#  this bar chart is a reliable alternative)

all_phrases = []
for text in texts:
    for p in extract_key_phrases(text):
        all_phrases.append(p['phrase'].lower())

phrase_series = pd.Series(all_phrases).value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(phrase_series.index[::-1], phrase_series.values[::-1],
               color='mediumpurple', edgecolor='white')
ax.set_title('Top 20 Key Phrases (noun chunks) Across Corpus', fontsize=13)
ax.set_xlabel('Frequency')
ax.bar_label(bars, padding=4, fontsize=9)
plt.tight_layout()
plt.show()

---
## 7. Document Section Categorisation
### 7a. Rule-based categoriser

In [ ]:
# ── Rule-based section classifier ─────────────────────────────────────────
#
# Rules (in priority order):
#   CAPTION : starts with Figure / Table / Chart / Appendix / Eq.
#   TITLE   : short (<= 12 tokens), no sentence-ending punctuation, title-cased
#   BODY    : everything else

import re

CAPTION_PATTERN = re.compile(
    r'^(figure|fig\.?|table|chart|appendix|eq\.?|equation)\s',
    re.IGNORECASE
)

def categorise_section(text):
    """Classify a document segment as title, caption, or body."""
    text_stripped = text.strip()
    
    # Rule 1 — Caption keywords at the start
    if CAPTION_PATTERN.match(text_stripped):
        return 'caption'
    
    # Rule 2 — Title: short, no full-stop ending, mostly title-cased words
    doc_obj      = nlp(text_stripped)
    token_count  = len([t for t in doc_obj if not t.is_space])
    ends_with_fs = text_stripped.endswith('.')
    words        = text_stripped.split()
    title_cased  = sum(1 for w in words if w[0].isupper()) / max(len(words), 1)
    
    if token_count <= 12 and not ends_with_fs and title_cased >= 0.5:
        return 'title'
    
    # Default — body paragraph
    return 'body'

# ── Apply to corpus and compare with ground truth ─────────────────────────
predictions = [categorise_section(t) for t in texts]

print(f'{"Doc":<5} {"Predicted":<12} {"Actual":<12} {"Match":<8} {"Text (first 60 chars)"}')
print('-' * 95)
correct = 0
for i, (pred, actual, text) in enumerate(zip(predictions, labels, texts)):
    match   = pred == actual
    correct += int(match)
    tick    = '✓' if match else '✗'
    preview = text[:60].replace('\n', ' ').replace('     ', ' ') + ('...' if len(text) > 60 else '')
    print(f'{i:<5} {pred:<12} {actual:<12} {tick:<8} {preview}')

print(f'\nAccuracy: {correct}/{len(texts)} = {correct/len(texts)*100:.0f}%')

### 7b. Visualise section categorisation

In [ ]:
# ── Plot predicted vs actual section labels ───────────────────────────────
category_colors = {'title': '#5B8CDB', 'caption': '#E5813A', 'body': '#4CAF82'}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, cat_list, title in zip(
    axes,
    [labels, predictions],
    ['Actual Section Labels', 'Predicted Section Labels']
):
    counts = pd.Series(cat_list).value_counts()
    colors = [category_colors[c] for c in counts.index]
    ax.pie(counts.values, labels=counts.index, colors=colors,
           autopct='%1.0f%%', startangle=140, textprops={'fontsize': 11})
    ax.set_title(title, fontsize=12)

plt.suptitle('Document Section Categorisation', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ── Per-document horizontal strip chart ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3.5))
y_pos = list(range(len(texts)))

for i, (pred, actual) in enumerate(zip(predictions, labels)):
    ax.barh(i - 0.18, 1, height=0.33, color=category_colors[actual],  alpha=0.85, label='Actual'   if i == 0 else '')
    ax.barh(i + 0.18, 1, height=0.33, color=category_colors[pred],    alpha=0.55, label='Predicted' if i == 0 else '')
    match_sym = '✓' if pred == actual else '✗'
    ax.text(1.05, i, match_sym, va='center', fontsize=12)

ax.set_yticks(y_pos)
ax.set_yticklabels([f'Doc {i}' for i in range(len(texts))], fontsize=9)
ax.set_xlim(0, 1.3)
ax.set_xticks([])
ax.set_title('Actual vs Predicted — per document (colour = category)', fontsize=11)

patches = [mpatches.Patch(color=v, label=k) for k, v in category_colors.items()]
patches += [mpatches.Patch(color='gray', alpha=0.85, label='Actual (solid)'),
            mpatches.Patch(color='gray', alpha=0.40, label='Predicted (faded)')]
ax.legend(handles=patches, loc='lower right', fontsize=8, framealpha=0.7)
plt.tight_layout()
plt.show()

---
## 8. Dependency Parsing (displaCy — from class code)
Applied to a key sentence from the corpus.

In [ ]:
# ── Dependency parse of a representative sentence ─────────────────────────
parse_sentence = (
    "Researchers at Google and OpenAI have published papers on named entity recognition."
)

doc_parse = nlp(parse_sentence)

html = displacy.render(
    doc_parse,
    style='dep',
    jupyter=False,          # same fix as used in class code
    options={
        'distance'      : 120,
        'color'         : 'blue',
        'arrow_stroke'  : 3,
        'arrow_spacing' : 20,
        'word_spacing'  : 45,
        'compact'       : True
    }
)

display(HTML(html))

---
## 9. Summary Printout — all deliverables in one view

In [ ]:
print('=' * 65)
print('PIPELINE SUMMARY')
print('=' * 65)

print('\n[0] IMAGE OCR INPUT')
print(f'    • Image file      : {IMAGE_FILE}')
print(f'    • Tesseract config: --oem 1 --psm 6')
ocr_words = len(ocr_text.split()) if ocr_text.strip() else 0
print(f'    • Words extracted : {ocr_words}')

print('\n[1] CLEANING PIPELINE')
print(f'    • Tokenisation    : spaCy en_core_web_sm')
print(f'    • Stopwords       : spaCy built-in list')
print(f'    • Lemmatisation   : spaCy (context-aware)')
print(f'    • Stemming shown  : NLTK PorterStemmer (comparison only)')

print('\n[2] TEXT REPRESENTATION')
bow_vocab = len(bow_vectorizer.get_feature_names_out())
print(f'    • BoW vocabulary  : {bow_vocab} terms (max_features=20)')
print(f'    • TF-IDF vocab    : {len(tfidf_vectorizer.get_feature_names_out())} terms (max_features=20)')

print('\n[3] NAMED ENTITY RECOGNITION')
print(f'    • Total entities  : {len(entities_df)}')
if not entities_df.empty:
    print(f'    • Entity types    : {", ".join(entities_df["type"].unique())}')
    top3 = entities_df["entity"].value_counts().head(3)
    print(f'    • Top entities    : {", ".join(top3.index.tolist())}')

print('\n[4] KEY PHRASE EXTRACTION')
top5_phrases = pd.Series(all_phrases).value_counts().head(5)
print(f'    • Top 5 phrases   : {", ".join(top5_phrases.index.tolist())}')

print('\n[5] SECTION CATEGORISATION')
print(f'    • Documents       : {len(texts)} (incl. OCR doc if loaded)')
print(f'    • Accuracy        : {correct}/{len(texts)} ({correct/len(texts)*100:.0f}%)')
print(f'    • Method          : Rule-based (regex + token count + title-case ratio)')

print('\n' + '=' * 65)